# Elastic Net Regression - California Housing Dataset

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

**Steps:**
1. **df.shape**: Check the number of rows (samples) and columns (features)
2. **df.info()**: Get information about data types, non-null counts, and memory usage
3. **df.describe()**: View statistical summary (mean, std, min, max, quartiles) for each feature

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

Missing values can significantly impact model performance. We need to identify and handle them appropriately.

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Standardization)

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## Elastic Net Regression Formula

**The Elastic Net Regression Equation:**

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n + \epsilon$$

**The Elastic Net Loss Function (with L1 and L2 Regularization):**

$$L(\beta) = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \lambda_1 \sum_{j=1}^{p} |\beta_j| + \lambda_2 \sum_{j=1}^{p} \beta_j^2$$

**Alternative Form (using mixing parameter):**

$$L(\beta) = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha [\rho \sum_{j=1}^{p} |\beta_j| + (1-\rho) \sum_{j=1}^{p} \beta_j^2]$$

Where:
- **y** = dependent variable (target/price)
- **x₁, x₂, ..., xₙ** = independent variables (features)
- **β₀, β₁, ..., βₙ** = regression coefficients
- **α (alpha)** = overall regularization parameter (controls total penalty strength)
- **ρ (l1_ratio)** = mixing parameter (0 ≤ ρ ≤ 1)
  - ρ = 0: Pure L2 regularization (Ridge)
  - ρ = 1: Pure L1 regularization (Lasso)
  - 0 < ρ < 1: Combination of L1 and L2
- **ε** = error term

**Key Concepts:**
- Elastic Net combines both L1 and L2 regularization penalties
- Balances the feature selection of Lasso with the stability of Ridge
- Particularly useful when:
  - Features are highly correlated
  - Number of features (p) is greater than number of samples (n)
  - You want some feature selection but not too aggressive
- The l1_ratio parameter controls the mix between L1 and L2:
  - Lower values: More like Ridge (keeps all features)
  - Higher values: More like Lasso (more feature selection)
- The alpha parameter controls the overall regularization strength

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Train Elastic Net with Default Parameters

In [ ]:
elastic_net = ElasticNet(alpha=1.0, l1_ratio=0.5)

elastic_net.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = elastic_net.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R²:",r2)

## Coefficient Analysis

In [ ]:
coef_df = pd.DataFrame({
    "Feature":X_train.columns,
    "Coefficient":elastic_net.coef_
})

coef_df.sort_values(
    by="Coefficient",
    ascending=False
)
print(
    "Intercept:",
    elastic_net.intercept_
)
print(
    "Non-zero coefficients:",
    np.sum(elastic_net.coef_ != 0)
)

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()]
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title(
    "Elastic Net Regression"
)

plt.show()

## Residual Plot

In [ ]:
residuals = y_test-y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(y=0)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Residuals"
)

plt.title(
    "Residual Plot"
)

plt.show()

## Hyperparameter Tuning: Alpha and L1 Ratio

We'll tune both parameters:
- **alpha**: Overall regularization strength
- **l1_ratio**: Mix between L1 and L2 (0 = pure Ridge, 1 = pure Lasso)

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

results = []

for alpha in alphas:
    for l1_ratio in l1_ratios:
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        r2 = r2_score(y_test, pred)
        
        results.append({
            'Alpha': alpha,
            'L1 Ratio': l1_ratio,
            'R²': r2,
            'Non-zero Coefs': np.sum(model.coef_ != 0)
        })

results_df = pd.DataFrame(results)
print(results_df.sort_values('R²', ascending=False).head(10))

## Visualize Hyperparameter Performance

In [ ]:
# Pivot for heatmap
pivot_r2 = results_df.pivot('Alpha', 'L1 Ratio', 'R²')
pivot_coefs = results_df.pivot('Alpha', 'L1 Ratio', 'Non-zero Coefs')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# R² heatmap
sns.heatmap(pivot_r2, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax1)
ax1.set_title('R² Score Heatmap')
ax1.set_xlabel('L1 Ratio')
ax1.set_ylabel('Alpha')

# Non-zero coefficients heatmap
sns.heatmap(pivot_coefs, annot=True, fmt='d', cmap='Blues', ax=ax2)
ax2.set_title('Non-zero Coefficients Heatmap')
ax2.set_xlabel('L1 Ratio')
ax2.set_ylabel('Alpha')

plt.tight_layout()
plt.show()

## Compare with Lasso and Ridge

Let's compare Elastic Net with pure Lasso and pure Ridge using the best parameters.

In [ ]:
from sklearn.linear_model import Lasso, Ridge

# Best parameters from our search
best_alpha = 0.01
best_l1_ratio = 0.1

# Elastic Net
en = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio)
en.fit(X_train, y_train)
y_pred_en = en.predict(X_test)
r2_en = r2_score(y_test, y_pred_en)

# Lasso (pure L1)
lasso = Lasso(alpha=best_alpha)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)

# Ridge (pure L2)
ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)

print("Comparison Results:")
print(f"Elastic Net (α={best_alpha}, l1_ratio={best_l1_ratio}): R² = {r2_en:.4f}, Non-zero coefs = {np.sum(en.coef_ != 0)}")
print(f"Lasso (α={best_alpha}): R² = {r2_lasso:.4f}, Non-zero coefs = {np.sum(lasso.coef_ != 0)}")
print(f"Ridge (α={best_alpha}): R² = {r2_ridge:.4f}, Non-zero coefs = {np.sum(ridge.coef_ != 0)}")

## Summary

Elastic Net regression provides:
- **Best of both worlds**: Combines L1's feature selection with L2's stability
- **Handles correlated features**: Better than Lasso when features are correlated
- **Flexible regularization**: Can tune the mix between L1 and L2 penalties
- **Robust performance**: Often outperforms pure Lasso or Ridge in practice

**When to use Elastic Net:**
- When you have many correlated features
- When p > n (more features than samples)
- When you want some feature selection but not too aggressive
- When you need to balance sparsity and stability